# CS 3110/5110: Data Privacy
## Homework 2

In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')
adult = adult.dropna()

## Question 3 (10 points)

Write code to determine the `Education-Num` of the individual named Ardyce Golby by performing a differencing attack. Your code should *only* use aggregate data to find Ardyce's education number.

In [10]:
def ardyce_education():
    # YOUR CODE HERE
    #adult[adult["Name"] == 'Brenn McNeely']['Age'].mean()
    result1 = adult[adult["Name"] != 'Ardyce Golby']['Education-Num'].mean()*(len(adult)-1)
    result2=adult['Education-Num'].mean()*len(adult)
    return result2 - result1
    raise NotImplementedError()

In [11]:
# TEST CASE for Question 3
assert ardyce_education() == 12

## Question 1 (20 points)

Implement a more efficient version of `is_k_anonymous`. The inefficient implementation, taken from the textbook, appears below.

**Hint**: use the [`value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html) or `group_by` functions, and make sure no count is less than $k$.

In [ ]:
# Checking for k-Anonymity, taken from the textbook
# def is_k_anonymous(k, qis, df):
#     for index, row in df.iterrows():
#         query = ' & '.join([f'`{col}` == "{row[col]}"' for col in qis])
#         rows = df.query(query)
#         if (rows.shape[0] < k):
#             return False
#     return True

In [30]:
# Checking for k-anonymity more efficiently
def is_k_anonymous(k, qis, df):
    """Returns true if df satisfies k-Anonymity for the quasi-identifiers 
    qis. Returns false otherwise."""
    values = df[qis].value_counts()
    for i in values.tolist():
        if i < k:
            return False
    
    return True

    # YOUR CODE HERE
    raise NotImplementedError()

In [31]:
# TEST CASES for question 1

assert not is_k_anonymous(2, ['Age'], adult)
assert is_k_anonymous(1, ['Age'], adult)
assert is_k_anonymous(1, ['Age', 'Occupation'], adult)

## Question 2 (10 points)

Consider the definition of `generalize` below, taken from the textbook. The function takes a dataframe `df` and a dictionary `depths` that describes how much to generalize each column of `df`. Generalizing a column to a depth of $n$ replaces the $n$ least-significant digits of each number in that column by zeroes. For example, we could generalize column `A` by making its least-significant digit a 0 and column `B` by doing the same for 2 digits with the following depth specification:

In [38]:
depths = {
    'A': 1,
    'B': 2
}

In [36]:
def generalize(df, depths):
    return df.apply(lambda x: x.apply(lambda y: int(int(y/(10**depths[x.name]))*(10**depths[x.name]))))

Using the `generalize` function, generalize the `Age` column of the `adult` dataset to a depth of 1. Drop the other columns of the dataset. Your result should achieve $k$-Anonymity for $k=20$.

In [39]:
def generalize_adult_age():
    depths = {
        # YOUR CODE HERE
        'Age' : 1
    }
    
    return generalize(adult[['Age']], depths)

In [40]:
assert is_k_anonymous(20, ['Age'], generalize_adult_age())

## Question 3 (10 points)

Using the `generalize` function, generalize the `Age` and `Zip` columns of the `adult` dataset in order to achieve $k$-Anonymity for $k=5$. Your result should drop other columns besides these two.

In [58]:
def generalize_adult_age_zip():
    depths = {
        'Age' : 1,
        'Zip' : 5
    }

    return generalize(adult[['Age', 'Zip']], depths)

In [59]:
assert is_k_anonymous(5, ['Age', 'Zip'], generalize_adult_age_zip())

## Question 4 (30 points)

In 1-4 sentences each, answer the following:

1. How much generalization was required to achieve $k=5$ in question 3?
2. Does this level of generalization significantly impact the utility of the $k$-Anonymized data? Why or why not?
3. Why is generalizing the `adult` dataset so challenging? (**Hint**: consider outliers)
4. Is there another approach, in addition to our simple generalization method, that might work better?
5. What is a simple method for generalizing the `Occupation` column?

1) In general, there had to be a decent amount of generalization to achive k-anonymity where k = 5 for the adult dataset. One solution that worked was adult generalized to a depth of 2 and zip generalized to a depth of 3. However, since this was essentially limiting any signifigance to age, I pushed for a secondary solution. Age to a depth of 1 and Zip to a depth of 5 also works, but again that eliminates all significance to the zip code being in the dataset (all zips are 00000). So, in order to achieve 5-anonymity for the adult dataset there needs to be signifigant generalization.
2) I believe this level of generalization does significantly impact the utility of the anonymized data. You are essentially asking what data needs to be reduced to a trivial column. if loctation is more important (tracking medical outcomes based on rural areas) prioritizing preserving zip code is going to be more impactful. If age is more important (geriatric health outcomes compared to the general populus) then prioritize preserving age. 
3) This dataset is difficult to generalize due mostly to the diversity of adults and people. there's likely many older individuals who are very few in number ie. 80+ years old that, when the data is generalized, there are still small groups with unique age + zip combinations. In that same vein, rural zip codes could only have a few people represented in the total dataset, and some zip codes don't even exist. smaller groups upon generalization are likely to happen, especially smaller combinations of rural towns
4) I believe that there's multiple other intuitive things that can be done to strengthen the anonymization of this data while preserving vital information. If outliers are minimal deleting a couple major outliers could help us generalize less aggressively to achieve k-anonymized data for a specific k. I also think adding noise into the dataset could be of use. when random entries are generated and added into the data then you can synthetically engineer anonymity while also preserving data without the need to delete. But, this is differential privacy and we'll get there I think.
5) For the occupation category I believe if slight generalization is needed you can group people by industry (unemployed, tech, healthcare, food service, etc.) if more major changes need to be made I think employed/unemployed could also work well.